In [ ]:
import pandas as pd
import math

In [ ]:
# contain meta data of prm 12k

correct_answer_df= pd.read_csv('./data/prm12k.csv')

## Generate self-distill questions datasets

In [ ]:
#build df
df_out = pd.DataFrame({
    "messages": correct_answer_df["question"].apply(
        lambda q: [{"role": "user", "content": q}]
    )
})

chunk_size = 995
num_chunks = math.ceil(len(df_out) / chunk_size)

for i in range(num_chunks):
    start = i * chunk_size
    end = (i + 1) * chunk_size

    chunk_df = df_out.iloc[start:end]

    output_path = f"self_distill_questions_dataset_{i+1}.jsonl"
    chunk_df.to_json(
        output_path,
        orient="records",
        lines=True,
        force_ascii=False
    )

    print(f"Saved {output_path} with {len(chunk_df)} records")

Use best_of_N_inference.sh to generate four self-distill datasets

## step 1

Self-distill datasets

In [ ]:
df_prm12k_seed1 = pd.read_json('self_distill_seed1.jsonl',lines=True)
df_prm12k_seed2 = pd.read_json('self_distill_seed2.jsonl',lines=True)
df_prm12k_seed3 = pd.read_json('self_distill_seed3.jsonl',lines=True)
df_prm12k_seed4 = pd.read_json('self_distill_seed4.jsonl',lines=True)

In [ ]:
import transformers
# ======  tokenizer ======
chat_tokenizer_dir = "./ds_tokenizer/"

tokenizer = transformers.AutoTokenizer.from_pretrained(
    chat_tokenizer_dir, trust_remote_code=True
)

def convert_to_good_format_df(df):
    df['question'] = df['messages'].apply(lambda x:x[0]['content'])
    #calculate tokens
    df['tokens'] = df['response'].apply(lambda x:token_calculator(x))
    df.drop(columns=['labels', 'logprobs'], errors='ignore', inplace=True)
    return df

def token_calculator(paragraphs):
    if isinstance(paragraphs, str):
        return len(tokenizer.encode(paragraphs))
    elif isinstance(paragraphs, list):
        return sum(len(tokenizer.encode(p)) for p in paragraphs if isinstance(p, str))
    else:
        return 0  # Fallback for unexpected types


In [ ]:
import pandas as pd

def select_and_merge_seeds(
    dfs: dict,
    shared_cols=("question",),
    per_seed_cols=("response", "tokens"),
):
    """
    dfs: {seed: DataFrame}
    shared_cols 
    per_seed_cols 
    """

    # 1️⃣ shared columns（只取 seed1）
    base_df = dfs[min(dfs.keys())][list(shared_cols)].copy()

    # 2️⃣ per-seed columns
    seed_dfs = []
    for seed, df in dfs.items():
        tmp = df[list(per_seed_cols)].copy()
        tmp = tmp.add_prefix(f"seed{seed}_")
        seed_dfs.append(tmp)

    # 3️⃣ column-wise concat
    return pd.concat([base_df] + seed_dfs, axis=1)

Use LLM as a judge to check the answer: inference.sh can be used to check answer

In [ ]:
import pandas as pd
from typing import List

def export_chat_df_judge(
    df: pd.DataFrame,
    seed: int,
    keep_cols: List[str] = None,
    tail_n: int = 100,
):
    if keep_cols is None:
        keep_cols = []

    response_col = f"seed{seed}_response"
    required_cols = ["target", response_col]

    missing = set(required_cols + keep_cols) - set(df.columns)
    assert not missing, f"Missing columns: {missing}"

    def build_messages(row):
        target = row["target"]
        response = str(row[response_col])[-tail_n:]

        PROMPT = f"""You are a strict answer verifier. Determine whether the model answer is correct.
If it is correct and in boxed format (e.g., boxed{{28}}), return True, else return False.

Correct Answer:
{target}

Model Answer:
{response}

Return True or False.
"""
        return [{"role": "user", "content": PROMPT}]

    df_out = pd.DataFrame()
    df_out["messages"] = df.apply(build_messages, axis=1)

    for col in keep_cols:
        df_out[col] = df[col]

    return df_out

In [ ]:
convert_to_good_format_df(df_prm12k_seed1)
convert_to_good_format_df(df_prm12k_seed2)
convert_to_good_format_df(df_prm12k_seed3)
convert_to_good_format_df(df_prm12k_seed4)

In [ ]:
dfs = {
    1: df_prm12k_seed1,
    2: df_prm12k_seed2,
    3: df_prm12k_seed3,
    4: df_prm12k_seed4,
}

df_merged = select_and_merge_seeds(
    dfs,
    shared_cols=("question",),
    per_seed_cols=("response", "tokens"),
)

In [ ]:
df_merged['target'] = df_merged['question'].map(
    correct_answer_df.set_index('question')['target']
)

## Step 2: verify answers

In [ ]:
for seed in [1, 2, 3, 4]:
    df_out = export_chat_df_judge(df_merged, seed=seed)
    df_out.to_json(
        f"self_distill_seed{seed}_check.jsonl",
        orient="records",
        lines=True
    )

df_prm12k_seed1_check_answer = pd.read_json('./data/self_distill_seed1_check_result.jsonl',lines=True)
df_prm12k_seed2_check_answer = pd.read_json('./data/self_distill_seed2_check_result.jsonl',lines=True)
df_prm12k_seed3_check_answer = pd.read_json('./data/self_distill_seed3_check_result.jsonl',lines=True)
df_prm12k_seed4_check_answer = pd.read_json('./data/self_distill_seed4_check_result.jsonl',lines=True)

In [ ]:
def str_to_bool(s):
    try:
        # 标准化字符串，处理大小写和空格
        s_lower = str(s).strip().lower()
        if s_lower in ['true', '1', 'yes']:
            return True
        elif s_lower in ['false', '0', 'no']:
            return False
        else:
            return False  # 无法识别的统一为 False
    except:
        return False

df_merged['seed1_TrueOrFalse'] = [str_to_bool(x) for x in df_prm12k_seed1_check_answer['response'].values]
df_merged['seed2_TrueOrFalse'] = [str_to_bool(x) for x in df_prm12k_seed2_check_answer['response'].values]
df_merged['seed3_TrueOrFalse'] = [str_to_bool(x) for x in df_prm12k_seed3_check_answer['response'].values]
df_merged['seed4_TrueOrFalse'] = [str_to_bool(x) for x in df_prm12k_seed4_check_answer['response'].values]

In [ ]:
import pandas as pd
import numpy as np

def get_best_seed(row):
    """
    row: DataFrame的一行数据，包含4个seed的column
    """
    valid_candidates = []
    
    # 1. 
    for i in range(1, 5): # Now loop from 1 to 4
        flag_col = f'seed{i}_TrueOrFalse'
        token_col = f'seed{i}_tokens'
        
        
        if row[flag_col] == True: 
            valid_candidates.append({
                'seed_name': f'seed{i}',
                'token_count': row[token_col]
            })
    

    if not valid_candidates:
        return 'Not exist'
    

    valid_candidates.sort(key=lambda x: x['token_count'])
    
    n = len(valid_candidates)
    
    
    target_index = n // 2
    best_seed = valid_candidates[target_index]['seed_name']
    
    return best_seed


In [ ]:
def extract_best_response(row):
    # 1. Get the previously computed best_seed (e.g., 'seed1', 'seed3', or 'Not exist')
    seed_name = row['best_seed']
    
    # 2. Guard clause: if no valid seed was found, return None (or 'Not exist')
    if seed_name == 'Not exist' or pd.isna(seed_name):
        return None  # or 'Not exist'
    
    # 3. Construct the target response column name
    #    e.g., 'seed1' + '_response' -> 'seed1_response'
    target_col = f"{seed_name}_response"
    
    # 4. Safety check to avoid KeyError due to missing columns
    #    (If the DataFrame is guaranteed to contain these columns,
    #     this check can be omitted.)
    if target_col in row.index:
        return row[target_col]
    else:
        # Column not found (e.g., only token-level data is available)
        return None

In [ ]:
df_merged['best_seed'] = df_merged.apply(get_best_seed, axis=1)
df_merged['best_seed_response'] = df_merged.apply(extract_best_response, axis=1)
df_merged_best_seed = df_merged[['question', 'best_seed_response','target']]
df_merged_best_seed_filtered= df_merged_best_seed[df_merged_best_seed['best_seed_response'].notna()].copy()

In [ ]:
import pandas as pd
import re

def extract_last_thinking_signal(text):
    """
    Logic:
    1. Locate the </think> tag to separate the thinking section.
    2. Search for **Final Answer** within the thinking section.
    3. If found, extract the content starting from that sentence
       up to the </think> boundary.
    4. If not found, return an empty string.
    """
    if not isinstance(text, str):
        return ''
        
    # 1. Split by </think> and keep only the first part
    parts = text.split('</think>')
    if len(parts) < 2:
        return 'No </think> Label'  # Missing </think> tag
    
    think_content = parts[0]
    
    # 2. Define the target phrase
    target_phrase = "**Final Answer**"
    
    # 3. Find the position of the target phrase
    idx = think_content.find(target_phrase)
    
    if idx == -1:
        return ''
    
    # 4. Smart backtracking: find the beginning of the sentence
    # We search backward for the nearest newline '\n' or period '.'
    # to avoid truncating prefixes such as
    # "Therefore, the **Final Answer** is ..."
    
    # Extract the text before idx to locate the last delimiter
    pre_text = think_content[:idx]
    
    # Find the nearest newline or period
    last_newline = pre_text.rfind('\n')
    last_period = pre_text.rfind('.')
    
    # Use the delimiter closest to the target phrase
    start_pos = max(last_newline, last_period)
    
    # If a delimiter is found, start from the character after it;
    # otherwise, start from the beginning
    if start_pos != -1:
        final_signal = think_content[start_pos + 1:]
    else:
        # The **Final Answer** appears at the very beginning
        final_signal = think_content[idx:]

    # 5. Trim leading and trailing whitespace
    return final_signal.strip()

def split_by_think_part(text: str, keep_tag: bool = False):
    """
    Split text into think part and remaining part by </think>.

    Args:
        text (str): full model output
        keep_tag (bool): whether to keep </think> in think_part

    Returns:
        think_part (str or None): content inside <think>...</think>
        rest_part (str): content after </think>
    """
    marker = "</think>"

    if marker not in text:
        return 'error' 

    idx = text.find(marker)
    end = idx + len(marker)

    think_part = text[:end] if keep_tag else text[:idx]
    rest_part = text[end:].lstrip()

    return think_part

def split_by_content_part(text: str, keep_tag: bool = False):
    """
    Split text into think part and remaining part by </think>.

    Args:
        text (str): full model output
        keep_tag (bool): whether to keep </think> in think_part

    Returns:
        think_part (str or None): content inside <think>...</think>
        rest_part (str): content after </think>
    """
    marker = "</think>"

    if marker not in text:
        return 'error'  # 没有 think，直接返回

    idx = text.find(marker)
    end = idx + len(marker)

    think_part = text[:end] if keep_tag else text[:idx]
    rest_part = text[end:].lstrip()

    return rest_part

Export best seed df

In [ ]:
df_merged_best_seed_filtered['last_thinking_sentence'] = df_merged_best_seed_filtered['best_seed_response'].apply(lambda x: extract_last_thinking_signal(x))
df_merged_best_seed_filtered['think'] = df_merged_best_seed_filtered['best_seed_response'].apply(lambda x: split_by_think_part(x))
df_merged_best_seed_filtered['content'] = df_merged_best_seed_filtered['best_seed_response'].apply(lambda x: split_by_content_part(x))
df_merged_best_seed_filtered['no_last_thinking_sentence'] = df_merged_best_seed_filtered['last_thinking_sentence'].apply(lambda x: x=='')
df_merged_best_seed_filtered.to_json('./data/self_distill_best_of_N_seed.jsonl',orient='records',lines=True)